# 다중 클래스 분류
- iris 데이터셋으로 다중 클래스 분류
- 손실함수 CCE
- 최종예측 argmax()

1. 데이터 불러오기

In [48]:
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris()

X = pd.DataFrame(
    iris.data,
    columns = iris.feature_names
)

y = pd.Series(
    iris.target,
    name = 'target'
)

print(X.shape)
print(y.value_counts().sort_index())
print(iris.target_names)

(150, 4)
target
0    50
1    50
2    50
Name: count, dtype: int64
['setosa' 'versicolor' 'virginica']


2. 학습 테스트 데이터 분리

In [49]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(120, 4) (30, 4) (120,) (30,)


3. 데이터 표준화

In [50]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

X_train_scaled

array([[-1.72156775, -0.33210111, -1.34572231, -1.32327558],
       [-1.12449223, -1.22765467,  0.41450518,  0.6517626 ],
       [ 1.14439475, -0.5559895 ,  0.58484978,  0.25675496],
       [-1.12449223,  0.11567567, -1.28894078, -1.45494479],
       [-0.40800161, -1.22765467,  0.13059752,  0.12508575],
       [ 0.54731923, -1.22765467,  0.69841284,  0.91510102],
       [-0.2885865 , -0.77987789,  0.24416059,  0.12508575],
       [ 0.54731923, -0.5559895 ,  0.75519438,  0.38842418],
       [ 2.21913069, -0.10821272,  1.3230097 ,  1.44177787],
       [ 2.21913069,  1.6828944 ,  1.66369889,  1.31010866],
       [ 2.09971558, -0.10821272,  1.60691736,  1.17843945],
       [ 0.18907392, -0.33210111,  0.41450518,  0.38842418],
       [-1.00507713, -2.34709662, -0.15331014, -0.26992188],
       [-0.04975629, -0.77987789,  0.18737906, -0.26992188],
       [-0.04975629, -1.00376628,  0.13059752, -0.00658346],
       [-1.36332244,  0.33956406, -1.23215924, -1.32327558],
       [-0.88566202,  1.

4. 텐서 변환

In [51]:
import torch

# 입력 데이터는 Pytorch Tensor float32 타입 사용
# X_train_tensor = torch.FloatTensor(X_train_scaled)와 동일
X_train_tensor = torch.tensor(X_train_scaled, dtype = torch.float32)    
X_test_tensor = torch.tensor(X_test_scaled, dtype = torch.float32)

# 분류 문제에서 CrossEntropyLoss 사용시 정답 라벨은 long 타입을 요구 (정수형)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype = torch.long)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype = torch.long)

print(X_train_tensor.size())
print(X_test_tensor.size())
print(y_train_tensor.size())
print(y_test_tensor.size())

torch.Size([120, 4])
torch.Size([30, 4])
torch.Size([120])
torch.Size([30])


In [52]:
y_train_tensor

tensor([0, 2, 1, 0, 1, 2, 1, 2, 2, 2, 2, 1, 1, 1, 1, 0, 0, 2, 2, 0, 1, 0, 2, 0,
        1, 2, 2, 0, 2, 0, 0, 1, 1, 0, 2, 2, 1, 1, 2, 1, 0, 1, 0, 2, 0, 0, 2, 0,
        0, 0, 0, 1, 2, 1, 0, 2, 1, 2, 0, 2, 0, 1, 2, 0, 1, 1, 2, 1, 1, 2, 0, 0,
        0, 2, 1, 2, 1, 2, 2, 1, 0, 2, 1, 0, 2, 0, 2, 1, 1, 0, 1, 2, 0, 0, 2, 2,
        2, 1, 2, 0, 2, 1, 2, 2, 0, 1, 1, 1, 1, 1, 0, 2, 1, 1, 0, 0, 0, 0, 1, 0])

In [53]:
y_test_tensor

tensor([0, 2, 1, 1, 0, 1, 0, 0, 2, 1, 2, 2, 2, 1, 0, 0, 0, 1, 1, 2, 0, 2, 1, 2,
        2, 1, 1, 0, 2, 0])

5. 모델 정의

In [54]:
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(X_train_tensor.size(-1), 16),    # 입력특성 4 -> 은닉층 노드 16개
    nn.ReLU(),                                 # 비선형성 추가

    nn.Linear(16,8),                           # 16 -> 8 은닉
    nn.ReLU(),

    nn.Linear(8,3)                             # 8 은닉 -> 최종 출력 3개 클래스
)

model

Sequential(
  (0): Linear(in_features=4, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=3, bias=True)
)

출력값 예시 : [0.2, 0.5, 0.3]              
가장 큰 값의 인덱스 :1                       
최종 예측 클래스 : versicolor                

6. 손실 함수와 옵티마이저 설정

In [55]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr = 0.01)

CrossEntropyLoss() : 모델의 출력값과 실제 클래스 번호를 비교해서 다중 클래스 분류 손실 계산           
CrossEntropyLoss()를 사용시에는 Softmax()를 따로 출력층에 붙이지 않는다.

7. 모델 학습

In [56]:
for epoch in range(1000):
    model.train()                            # 학습 모드로 설정

    optimizer.zero_grad()                    # 이전 step 기울기 초기화
    output = model(X_train_tensor)           # 순전파 : 결과 예측
    loss = criterion(output, y_train_tensor) # 손실 계산

    loss.backward()                          # 기울기 계산
    optimizer.step()                         # 가중치 업데이트
 
    # 100번마다 출력
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}의 loss : {loss.item():.4f}") #python float형태로 출력

Epoch 100의 loss : 0.0360
Epoch 200의 loss : 0.0192
Epoch 300의 loss : 0.0034
Epoch 400의 loss : 0.0011
Epoch 500의 loss : 0.0006
Epoch 600의 loss : 0.0003
Epoch 700의 loss : 0.0002
Epoch 800의 loss : 0.0002
Epoch 900의 loss : 0.0001
Epoch 1000의 loss : 0.0001


8. 출력값 확인

In [57]:
model.eval()                              # 평가 모드로 전환

with torch.no_grad():                 
    output = model(X_test_tensor)         # 테스트 데이터로 예측값 계산

print(output[:5])                         # 확률이 아닌 클래스별 원시 점수(로짓) 출력
print(output.shape)    

tensor([[  6.5772, -15.7438,  -9.7677],
        [-10.2506,  -5.8124,   6.5636],
        [ -5.6469,   3.3245,  -5.4130],
        [ -7.0154,   4.6618,  -4.9428],
        [  7.1443, -17.0058,  -9.9100]])
torch.Size([30, 3])


9.클래스 예측

In [58]:
with torch.no_grad():
    output = model(X_test_tensor)  # 테스트 데이터로 예측값 계산
    prediction = output.argmax(dim = 1) # 가장 높은 점수의 클래스


print(output[:10])
print(prediction[:10])    

tensor([[  6.5772, -15.7438,  -9.7677],
        [-10.2506,  -5.8124,   6.5636],
        [ -5.6469,   3.3245,  -5.4130],
        [ -7.0154,   4.6618,  -4.9428],
        [  7.1443, -17.0058,  -9.9100],
        [ -5.4028,   6.8667,  -2.3621],
        [  7.0725, -17.4574, -10.6522],
        [  2.5750,  -7.9707,  -9.9744],
        [-14.2467, -15.5826,  13.4529],
        [ -5.6586,   7.3190,  -2.7388]])
tensor([0, 2, 1, 1, 0, 1, 0, 0, 2, 1])


10. 확률값 확인

In [59]:
with torch.no_grad():
    output = model(X_test_tensor)  # 테스트 데이터로 예측값 계산
    probability = torch.softmax(output, dim = 1) # 클래스별 확률 변환
    prediction = output.argmax(dim = 1) # 가장 높은 점수의 클래스


print(output[:10])                               # 클래스별 로짓값
# 값이 여러개인 Tensor에서 
for row in probability[:10]:                
    print([f"{p.item():.6f}" for p in row])      # 클래스별 확률값
print(prediction[:10])                            # 예측 클래스

tensor([[  6.5772, -15.7438,  -9.7677],
        [-10.2506,  -5.8124,   6.5636],
        [ -5.6469,   3.3245,  -5.4130],
        [ -7.0154,   4.6618,  -4.9428],
        [  7.1443, -17.0058,  -9.9100],
        [ -5.4028,   6.8667,  -2.3621],
        [  7.0725, -17.4574, -10.6522],
        [  2.5750,  -7.9707,  -9.9744],
        [-14.2467, -15.5826,  13.4529],
        [ -5.6586,   7.3190,  -2.7388]])
['1.000000', '0.000000', '0.000000']
['0.000000', '0.000004', '0.999996']
['0.000127', '0.999713', '0.000160']
['0.000008', '0.999924', '0.000067']
['1.000000', '0.000000', '0.000000']
['0.000005', '0.999897', '0.000098']
['1.000000', '0.000000', '0.000000']
['0.999970', '0.000026', '0.000004']
['0.000000', '0.000000', '1.000000']
['0.000002', '0.999955', '0.000043']
tensor([0, 2, 1, 1, 0, 1, 0, 0, 2, 1])


모델에 Softmax()함수는 사용하지 않는다.
왜냐하면 CrossEntropyLoss에서 내부적으로 계산하기 때문에

11. 모델 평가

In [63]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = prediction.numpy()
y_true = y_test_tensor.numpy()

print(f"정확도 : {accuracy_score(y_pred, y_true)}")
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=iris.target_names))

정확도 : 0.9666666666666667
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

